# 02 — Mortality Risk Modeling

**Task:** predict `hospital_expire_flag` (in-hospital mortality) from demographics, comorbidity
burden, and the first 24 hours of ICU vitals/labs — an early-warning-score-style task, evaluated
the way clinical risk models are actually evaluated: **AUROC and AUPRC as the headline metrics,
not accuracy**, since mortality is a rare/imbalanced outcome (see prevalence in `01_EDA.ipynb`) —
a model that always predicts "survived" would score high accuracy while being clinically useless.

Run `python src/pipeline.py` first.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier

from config import PROCESSED_DIR, RANDOM_STATE
from pipeline import TARGET_COL

train = pd.read_csv(PROCESSED_DIR / "train.csv")
test = pd.read_csv(PROCESSED_DIR / "test.csv")

FEATURE_COLS = [c for c in train.columns if c not in (TARGET_COL, "stay_id")]
X_train, y_train = train[FEATURE_COLS], train[TARGET_COL]
X_test, y_test = test[FEATURE_COLS], test[TARGET_COL]
print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())

## Baseline: Logistic Regression

Chosen as the baseline specifically because it's interpretable and directly comparable in spirit
to clinically-trusted scoring systems like SAPS/APACHE — a black-box model that merely beats it
needs to justify the interpretability it gives up.

In [ ]:
logreg = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_test)[:, 1]

print(f"Logistic Regression — AUROC: {roc_auc_score(y_test, logreg_proba):.3f}, "
      f"AUPRC: {average_precision_score(y_test, logreg_proba):.3f}")

## Random Forest & XGBoost

In [ ]:
rf = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)),
])
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
print(f"Random Forest — AUROC: {roc_auc_score(y_test, rf_proba):.3f}, "
      f"AUPRC: {average_precision_score(y_test, rf_proba):.3f}")

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
)
xgb.fit(X_train, y_train)
xgb_proba = xgb.predict_proba(X_test)[:, 1]
print(f"XGBoost — AUROC: {roc_auc_score(y_test, xgb_proba):.3f}, "
      f"AUPRC: {average_precision_score(y_test, xgb_proba):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name, proba in [("LogReg", logreg_proba), ("RandomForest", rf_proba), ("XGBoost", xgb_proba)]:
    RocCurveDisplay.from_predictions(y_test, proba, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title("ROC — in-hospital mortality")
plt.show()

## Calibration

A risk score that clinicians act on needs to be calibrated (a predicted 20% risk should mean
roughly 1-in-5 such patients die), not just well-ranked — AUROC alone doesn't check this.

In [ ]:
best_proba = xgb_proba  # swap for whichever model has the best AUPRC once run
prob_true, prob_pred = calibration_curve(y_test, best_proba, n_bins=10, strategy="quantile")

plt.plot(prob_pred, prob_true, marker="o", label="model")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="perfectly calibrated")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.legend()
plt.title("Calibration curve")
plt.show()

## SHAP: explainability

For a clinical model, feature importance needs to be sanity-checked against clinical plausibility,
not just reported — e.g. lactate, GCS, and age topping the ranking would be expected; an
administrative field topping it would be a red flag for leakage.

In [ ]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

shap.summary_plot(shap_values, X_test, max_display=20)

In [ ]:
import joblib

joblib.dump(xgb, PROCESSED_DIR / "model_xgb.joblib")
print("Saved model for use in 03_Fairness_Audit.ipynb")